#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
CATENETS_DIR = ROOT / "experiments" / "supplementary" / "catenets_custom"
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# install dependencies - only once
%pip install wandb loguru jax ott-jax gdown
%pip install -e "{CATENETS_DIR}"

In [ ]:
# custom library imports - restart kernel after installation if needed
from catenets.models.torch.ranknet import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def init_rank_learner(input_dim, params, seed):
    return RankLearner(
        # standard init
        n_unit_in = input_dim,
        binary_y = False,
        n_iter_print = 1,
        seed=seed,

        # model capacity
        n_layers_out_nuis = 2,
        n_units_out_nuis = params['hidden_dim_nuis'],
        n_layers_r_nuis = 2,
        n_units_r_nuis = params['hidden_dim_nuis'],

        n_layers_out_rank = 2,
        n_units_out_rank = params['hidden_dim_rank'],
        n_layers_r_rank = 2,
        n_units_r_rank = params['hidden_dim_rank'],
        
        # tuned parameters
        weight_decay = params['weight_decay'],
        lr = params['learning_rate'],
        batch_size= params['batch_size'],
        kappa = params['kappa'],

        # early stopping
        early_stopping=True,
        n_iter = 50,
        nuisance_epochs=50,
        ranker_epochs=50,
        n_iter_min = 5, 
        patience = 5,
        val_split_prop=0.2,

        # model-specific settings
        nonlin='elu',
        dropout=False,
        dropout_prob = 0.0,
        batch_norm = True,
        orthogonal_weight = 1.0,
        clip_labels=True,
        pair_batch_size = params['pair_batch_size'])

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/rank_learner.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim_nuis=int(row['hidden_dim_nuis']),
    hidden_dim_rank=int(row['hidden_dim_rank']),
    pair_batch_size=int(row['pair_batch_size']),
    learning_rate=float(row['lr']),
    kappa=float(row['kappa']),
    weight_decay=float(row['weight_decay']),
    batch_size=int(row['batch_size']))

In [ ]:
# set directories
out_dir = f'./chkpts/'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"size_{train_size}" / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data - for both stages
    train_df_ns, val_df_ns, train_df_rank, val_df_rank, test_df = make_splits(df=df, train_size=train_size, seed=seed)

    # init model
    model = init_rank_learner(input_dim, params, seed)

    # STAGE 1: nuisances
    train_df_ns = pd.concat([train_df_ns, val_df_ns])
    X = train_df_ns[confounders].to_numpy(dtype=np.float32)
    t = train_df_ns["T"].to_numpy(dtype=np.float32)   
    y = train_df_ns["Y"].to_numpy(dtype=np.float32)
    model, best_loss = model.fit_nuisance(X, y, t)
    
    # STAGE 2: ranker
    train_df_rank = pd.concat([train_df_rank, val_df_rank])
    X = train_df_rank[confounders].to_numpy(dtype=np.float32)
    t = train_df_rank["T"].to_numpy(dtype=np.float32)   
    y = train_df_rank["Y"].to_numpy(dtype=np.float32)
    tau, dr = model.prepare_ranker_targets(X, y, t)
    model, best_autoc = model.fit_ranker(X, tau, dr)
    
    # checkpoint
    torch.save(model.state_dict(), ckpt_dir / "RankLearner.pt")

#### evaluation

In [ ]:
def load_rank_learner(train_size, params, seed, confounders, device):
    input_dim = len(confounders)

    # set checkpoint path
    ckpt_path = ROOT / "experiments" / "supplementary" / "baselines" / "catenets_baselines" / "chkpts" / f"size_{train_size}" / f"seed_{seed}" / "RankLearner.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    # load model checkpoint
    model = init_rank_learner(input_dim=input_dim, params=params, seed=seed)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    model.eval()
    
    return model

In [ ]:
def get_estimates_rank_learner(model, confounders, test_df):
    X_test = test_df[confounders].to_numpy(dtype=np.float32)
    score_test = model.predict(X_test).reshape(-1).detach().cpu().numpy()
    test_df["ranker_hat"] = score_test
    return test_df

In [ ]:
def compute_metrics_rank_learner(eval_df):
    required = {"ranker_hat", "cate"}

    # check if estimates are available
    missing = required - set(eval_df.columns)
    if missing:
        raise ValueError(f"eval_df is missing required columns: {sorted(missing)}")

    specs = [("RankLearner", "ranker_hat")]
    rows = []
    for name, score_col in specs:
        ranked = eval_df.sort_values(score_col, ascending=False).copy()

        rows.append({
            "model": name,
            "autoc": autoc(ranked),
            "policy_value": policy_value(ranked)})

    return pd.DataFrame(rows)

In [ ]:
# init collector
all_metrics = []

# loop over seeds and sizes
for seed in range(5):
    for size in [100, 250, 500, 1000, 2000]:

        # get testing data
        _, _, _, _, test_df = make_splits(df=df, train_size=size, seed=seed)

        # load model
        rank_learner = load_rank_learner(size, params, seed, confounders, device)
        df_eval = get_estimates_rank_learner(rank_learner, confounders, test_df)
        df_metrics = compute_metrics_rank_learner(df_eval)

        # store
        df_metrics["size"] = size
        df_metrics["seed"] = seed
        all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
summary = (df_all.groupby(["size", 'model']).agg(['mean', 'std']).reset_index())